In [2]:
import pandas as pd
import numpy as np
from urllib.parse import quote

import json
import re
from tqdm.auto import tqdm

from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

In [3]:
# Load the dataset
data = pd.read_csv('../../data/kc_dataset/metacademy/metacademy-prerequisite-pairs-transoformed-to-wikipedia.csv')
print(data.info())
data.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3201 entries, 0 to 3200
Data columns (total 4 columns):
 #   Column                        Non-Null Count  Dtype 
---  ------                        --------------  ----- 
 0   prereq_wiki_canonical_title   3201 non-null   object
 1   MA_preq                       3201 non-null   object
 2   MA_concept                    3201 non-null   object
 3   concept_wiki_canonical_title  3201 non-null   object
dtypes: object(4)
memory usage: 100.2+ KB
None


,prereq_wiki_canonical_title,MA_preq,MA_concept,concept_wiki_canonical_title
0,Set (mathematics),countable sets,cardinality,Cardinality
1,Set (mathematics),countable sets,cardinality,Infinity
2,Set (mathematics),countable sets,completeness of first-order logic,First-order logic
3,Set (mathematics),countable sets,completeness of first-order logic,G�del's completeness theorem
4,Set (mathematics),countable sets,compactness of propositional logic,Compactness theorem


In [4]:
REPL_CHAR = "\ufffd"  # �

def show_bad_values(df: pd.DataFrame, col: str):
    s = df[col].dropna().astype(str)
    bad = s[s.str.contains(REPL_CHAR, regex=False)].drop_duplicates().sort_values()
    print(f"\n=== {col}: '�' 포함 유니크 값 ({len(bad)}개) ===")
    if len(bad) == 0:
        print("(없음)")
    else:
        print(bad.to_string(index=False))

# data에서 확인
show_bad_values(data, "concept_wiki_canonical_title")
show_bad_values(data, "prereq_wiki_canonical_title")

## 실제 값: Alpha–beta pruning, Gödel's completeness theorem


=== concept_wiki_canonical_title: '�' 포함 유니크 값 (2개) ===
          Alpha�beta pruning
G�del's completeness theorem

=== prereq_wiki_canonical_title: '�' 포함 유니크 값 (1개) ===
G�del's completeness theorem


In [5]:
import re

def show_non_ascii_values(df: pd.DataFrame, col: str):
    s = df[col].dropna().astype(str)
    non_ascii = s[s.str.contains(r"[^\x00-\x7F]", regex=True)].drop_duplicates().sort_values()
    print(f"\n=== {col}: non-ASCII 포함 유니크 값 ({len(non_ascii)}개) ===")
    if len(non_ascii) == 0:
        print("(없음)")
    else:
        print(non_ascii.to_string(index=False))

show_non_ascii_values(data, "concept_wiki_canonical_title")
show_non_ascii_values(data, "prereq_wiki_canonical_title")



=== concept_wiki_canonical_title: non-ASCII 포함 유니크 값 (3개) ===
             Alpha�beta pruning
Gödel's incompleteness theorems
   G�del's completeness theorem

=== prereq_wiki_canonical_title: non-ASCII 포함 유니크 값 (1개) ===
G�del's completeness theorem


### Node
```
{
  "id": int,
  "name": "string",
  "category": ["string"],
  "link": "string", 
  "alias": [List]
}
```

In [6]:
## 카테고리는 존재 X 위키피디아 덤프로 넣어야 함
## 링크의 경우 https://en.wikipedia.org/wiki/이름 (띄어씌는 _로 변경됨)
## id는 그냥 GDB에서 매핑되기 때문에 필요 X

### Edge

```
A -{PREREQ}→ B
```
- A는 B의 선수 지식이다.
- B를 이해하기 위해 A가 필요하다.
```
{
  "id": int,
  "strength": float,
  "reason": "string"
}
```

In [7]:
## strength, reason은 LLM으로 생성해야 함

### 전처리 데이터셋 구축
#### 칼럼
- name
- link
- prereq_name

- 구축 방식: 
    - 이렇게 만들어놓은 다음 strength, reason, alias 생성
    - 생성되면 일단 노드 구축
    - prereq_name을 활용해 엣지 연결

In [8]:
WIKI_PREFIX = "https://en.wikipedia.org/wiki/"

# � 매핑
fix_map = {
    "Alpha\uFFFDbeta pruning": "Alpha–beta pruning",              # en dash
    "G\uFFFDdel's completeness theorem": "Gödel's completeness theorem",  # ö
}

data_clean = data.copy()
data_clean["concept_wiki_canonical_title"] = data_clean["concept_wiki_canonical_title"].replace(fix_map)
data_clean["prereq_wiki_canonical_title"] = data_clean["prereq_wiki_canonical_title"].replace(fix_map)

# concept별 prereq 리스트 만들기 (중복 제거 + 정렬 + 결측 제거)
concept_prereq_map = (
    data_clean.groupby("concept_wiki_canonical_title")["prereq_wiki_canonical_title"]
        .apply(lambda s: sorted(set(s.dropna().astype(str).map(str.strip))))
        .reset_index()
        .rename(columns={
            "concept_wiki_canonical_title": "name",
            "prereq_wiki_canonical_title": "prereq_name",
        })
)

# link 만들기: https://en.wikipedia.org/wiki/<name> (공백은 _)
#    + URL 인코딩해서 – / ö 같은 문자도 안전하게 처리
def make_wiki_link(name: str):
    if pd.isna(name):
        return pd.NA
    name = str(name).strip()
    if not name:
        return pd.NA
    return WIKI_PREFIX + quote(name.replace(" ", "_"), safe="()'_-")

concept_prereq_map["link"] = concept_prereq_map["name"].apply(make_wiki_link)

# preprocess_data 새로 생성
preprocess_data = concept_prereq_map[["name", "link", "prereq_name"]].copy()

In [9]:
preprocess_data.iloc[0]

name                                        A priori probability
link           https://en.wikipedia.org/wiki/A_priori_probabi...
prereq_name    [Erlang distribution, Gamma distribution, Inve...
Name: 0, dtype: object

In [10]:
data[data['concept_wiki_canonical_title'] == 'A priori probability']

,prereq_wiki_canonical_title,MA_preq,MA_concept,concept_wiki_canonical_title
1874,Gamma distribution,gamma distribution,uninformative priors,A priori probability
1877,Erlang distribution,gamma distribution,uninformative priors,A priori probability
1880,Inverse-gamma distribution,gamma distribution,uninformative priors,A priori probability
1883,Normal-gamma distribution,gamma distribution,uninformative priors,A priori probability


In [11]:
preprocess_data_list = concept_prereq_map[["name", "link", "prereq_name"]].copy()

# prereq_name을 행으로 펼친 preprocess_data (concept-prereq 한 쌍 = 한 행)
preprocess_data = (
    preprocess_data_list
    .explode("prereq_name", ignore_index=True)     # 리스트를 행으로 펼침
    .rename(columns={"prereq_name": "prereq_name"})     # 컬럼명 취향대로
)

preprocess_data

,name,link,prereq_name
0,A priori probability,https://en.wikipedia.org/wiki/A_priori_probabi...,Erlang distribution
1,A priori probability,https://en.wikipedia.org/wiki/A_priori_probabi...,Gamma distribution
2,A priori probability,https://en.wikipedia.org/wiki/A_priori_probabi...,Inverse-gamma distribution
3,A priori probability,https://en.wikipedia.org/wiki/A_priori_probabi...,Normal-gamma distribution
4,AC-3 algorithm,https://en.wikipedia.org/wiki/AC-3_algorithm,Asymptotic computational complexity
...,...,...,...
2911,Z-test,https://en.wikipedia.org/wiki/Z-test,Variance
2912,Z-test,https://en.wikipedia.org/wiki/Z-test,Z-test
2913,Zero of a function,https://en.wikipedia.org/wiki/Zero_of_a_function,Complex analysis
2914,Zero of a function,https://en.wikipedia.org/wiki/Zero_of_a_function,Complex number


In [12]:
## name == prereq_name 인 거 제거
preprocess_data = preprocess_data.loc[preprocess_data["name"] != preprocess_data["prereq_name"]].copy()

## name에 없는 prereq_name가 있다면 name으로 등록 후 prereq_name none

names = set(preprocess_data["name"].dropna().astype(str))
prereqs = set(preprocess_data["prereq_name"].dropna().astype(str))

missing_as_names = sorted(prereqs - names)

new_rows = pd.DataFrame({
    "name": missing_as_names,
    "link": [make_wiki_link(x) for x in missing_as_names],
    "prereq_name": [pd.NA] * len(missing_as_names),
})

preprocess_data = pd.concat([preprocess_data, new_rows], ignore_index=True)
print(f"추가된 신규 name 개수: {len(missing_as_names)}")

추가된 신규 name 개수: 55


In [13]:
preprocess_data

,name,link,prereq_name
0,A priori probability,https://en.wikipedia.org/wiki/A_priori_probabi...,Erlang distribution
1,A priori probability,https://en.wikipedia.org/wiki/A_priori_probabi...,Gamma distribution
2,A priori probability,https://en.wikipedia.org/wiki/A_priori_probabi...,Inverse-gamma distribution
3,A priori probability,https://en.wikipedia.org/wiki/A_priori_probabi...,Normal-gamma distribution
4,AC-3 algorithm,https://en.wikipedia.org/wiki/AC-3_algorithm,Asymptotic computational complexity
...,...,...,...
2898,Unambiguous finite automaton,https://en.wikipedia.org/wiki/Unambiguous_fini...,<NA>
2899,Variable elimination,https://en.wikipedia.org/wiki/Variable_elimina...,<NA>
2900,Vector space,https://en.wikipedia.org/wiki/Vector_space,<NA>
2901,Vector spaces without fields,https://en.wikipedia.org/wiki/Vector_spaces_wi...,<NA>


### node, edge 1차 데이터셋 생성

In [14]:
## node
## 칼럼: name, link, alias
nodes = (
    preprocess_data[["name", "link"]]
    .dropna(subset=["name"])
    .drop_duplicates(subset=["name"], keep="first")
    .copy()
)
nodes["alias"] = pd.NA


## edge
## 칼럼: name, prereq_name, strength, reason
edges = (
    preprocess_data[["name", "prereq_name"]]
    .dropna(subset=["name", "prereq_name"])
    .copy()
)
# 중복 엣지 제거(같은 (name, prereq_name) 여러 번 있으면 1개만)
edges = edges.drop_duplicates(subset=["name", "prereq_name"]).reset_index(drop=True)

edges["strength"] = pd.NA
edges["reason"] = pd.NA

nodes = nodes.sort_values("name").reset_index(drop=True)
edges = edges.sort_values(["name", "prereq_name"]).reset_index(drop=True)

In [15]:
nodes

,name,link,alias
0,A priori probability,https://en.wikipedia.org/wiki/A_priori_probabi...,<NA>
1,AC-3 algorithm,https://en.wikipedia.org/wiki/AC-3_algorithm,<NA>
2,Abstract data type,https://en.wikipedia.org/wiki/Abstract_data_type,<NA>
3,Abstract type,https://en.wikipedia.org/wiki/Abstract_type,<NA>
4,Adjusted mutual information,https://en.wikipedia.org/wiki/Adjusted_mutual_...,<NA>
...,...,...,...
467,Well-quasi-ordering,https://en.wikipedia.org/wiki/Well-quasi-ordering,<NA>
468,Women in government,https://en.wikipedia.org/wiki/Women_in_government,<NA>
469,Y-fast trie,https://en.wikipedia.org/wiki/Y-fast_trie,<NA>
470,Z-test,https://en.wikipedia.org/wiki/Z-test,<NA>


In [16]:
edges

,name,prereq_name,strength,reason
0,A priori probability,Erlang distribution,<NA>,<NA>
1,A priori probability,Gamma distribution,<NA>,<NA>
2,A priori probability,Inverse-gamma distribution,<NA>,<NA>
3,A priori probability,Normal-gamma distribution,<NA>,<NA>
4,AC-3 algorithm,Asymptotic computational complexity,<NA>,<NA>
...,...,...,...,...
2843,Z-test,Uniform distribution (continuous),<NA>,<NA>
2844,Z-test,Variance,<NA>,<NA>
2845,Zero of a function,Complex analysis,<NA>,<NA>
2846,Zero of a function,Complex number,<NA>,<NA>


In [17]:
## 엣지에 있는 name, prereq_name이 노드에도 존재하는지 확인

node_names = set(nodes["name"].dropna().astype(str))
edge_names = set(edges["name"].dropna().astype(str))
edge_prereqs = set(edges["prereq_name"].dropna().astype(str))

# 노드에 없는 것 찾기
missing_edge_names = sorted(edge_names - node_names)
missing_edge_prereqs = sorted(edge_prereqs - node_names)

print(f"edges.name 중 nodes에 없는 값: {len(missing_edge_names)}개")

print(f"\nedges.prereq_name 중 nodes에 없는 값: {len(missing_edge_prereqs)}개")

edges.name 중 nodes에 없는 값: 0개

edges.prereq_name 중 nodes에 없는 값: 0개


### strength, reason, alias 생성
- 사용 모델: JunHowie/Qwen3-30B-A3B-Instruct-2507-GPTQ-Int4

In [18]:
MODEL = "JunHowie/Qwen3-30B-A3B-Instruct-2507-GPTQ-Int4"  ## "Qwen/Qwen3-30B-A3B-GPTQ-Int4"

tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)

llm = LLM(
    model=MODEL,
    trust_remote_code=True,
    max_model_len=2048,
    gpu_memory_utilization=0.9,
    tensor_parallel_size=1,
    enable_prefix_caching=True,
    enforce_eager=True,
)

sampling = SamplingParams(
    temperature=0.2,
    top_p=0.9,
    max_tokens=256,
)

def extract_json(text: str):
    if text is None:
        return None
    t = text.strip()
    if not t:
        return None

    # <think>...</think> 제거
    t = re.sub(r"<think>[\s\S]*?</think>", "", t).strip()

    # 코드블록(JSON) 우선 시도
    m = re.search(r"```(?:json)?\s*([\s\S]*?)\s*```", t, flags=re.IGNORECASE)
    if m:
        cand = m.group(1).strip()
        # (a) 그대로
        try:
            return json.loads(cand)
        except Exception:
            # (b) 흔한 누락 보정: 마지막 } 없으면 붙여보기
            c2 = cand
            if c2.startswith("{") and not c2.endswith("}"):
                try:
                    return json.loads(c2 + "}")
                except Exception:
                    pass

    # 텍스트에서 JSON 객체 구간 추출
    start = t.find("{")
    if start == -1:
        return None

    # 마지막 }가 있으면 그 구간으로 파싱, 없으면 끝까지를 후보로
    end = t.rfind("}")
    if end != -1 and end > start:
        cand = t[start:end+1].strip()
        try:
            return json.loads(cand)
        except Exception:
            # 마지막 }가 있었는데도 실패하면, 혹시 뒤에 군더더기/잘림 대비 보정도 시도
            pass
    else:
        cand = t[start:].strip()

    # (a) 그대로
    try:
        return json.loads(cand)
    except Exception:
        pass

    # (b) 마지막 } 누락 보정
    if cand.startswith("{") and not cand.endswith("}"):
        try:
            return json.loads(cand + "}")
        except Exception:
            pass

    # (c) 마지막 ]까진 있는데 }만 없는 형태 보정: ...]  -> ...]}
    if cand.startswith("{") and not cand.endswith("}") and cand.rstrip().endswith("]"):
        try:
            return json.loads(cand.rstrip() + "}")
        except Exception:
            pass

    return None


def build_chat_prompt(system: str, user: str) -> str:
    msgs = [{"role": "system", "content": system},
            {"role": "user", "content": user}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

INFO 01-25 16:59:23 [utils.py:263] non-default args: {'trust_remote_code': True, 'max_model_len': 2048, 'enable_prefix_caching': True, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'JunHowie/Qwen3-30B-A3B-Instruct-2507-GPTQ-Int4'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 01-25 16:59:25 [model.py:530] Resolved architecture: Qwen3MoeForCausalLM
WARNING 01-25 16:59:25 [model.py:1817] Your device 'Tesla V100-SXM2-32GB' (with compute capability 7.0) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 01-25 16:59:25 [model.py:1869] Casting torch.bfloat16 to torch.float16.
INFO 01-25 16:59:25 [model.py:1545] Using max model len 2048


2026-01-25 16:59:27,577	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 01-25 16:59:27 [scheduler.py:229] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 01-25 16:59:27 [gptq.py:99] Currently, the 4-bit gptq_gemm kernel for GPTQ is buggy. Please switch to gptq_marlin or gptq_bitblas.


...

INFO 01-25 16:59:36 [vllm.py:630] Asynchronous scheduling is enabled.
INFO 01-25 16:59:36 [vllm.py:637] Disabling NCCL for DP synchronization when using async scheduling.
WARNING 01-25 16:59:36 [vllm.py:665] Enforce eager set, overriding optimization level to -O0
INFO 01-25 16:59:36 [vllm.py:765] Cudagraph is disabled under eager mode
WARNING 01-25 16:59:38 [system_utils.py:136] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(EngineCore_DP0 pid=41855) INFO 01-25 16:59:46 [core.py:97] Initializing a V1 LLM engine (v0.14.0) with config: model='JunHowie/Qwen3-30B-A3B-Instruct-2507-GPTQ-Int4', speculative_config=None, tokenizer='JunHowie/Qwen3-30B-A3B-Instruct-2507-GPTQ-Int4', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch

(EngineCore_DP0 pid=41855) /data/ephemeral/home/8054_charmi/PTMT-GraphDB/.venv/lib/python3.10/site-packages/tvm_ffi/_optional_torch_c_dlpack.py:174: UserWarning: Failed to JIT torch c dlpack extension, EnvTensorAllocator will not be enabled.
(EngineCore_DP0 pid=41855) We recommend installing via `pip install torch-c-dlpack-ext`
(EngineCore_DP0 pid=41855)   warnings.warn(


(EngineCore_DP0 pid=41855) INFO 01-25 16:59:50 [cuda.py:351] Using TRITON_ATTN attention backend out of potential backends: ('TRITON_ATTN', 'FLEX_ATTENTION')


Loading safetensors checkpoint shards:   0% Completed | 0/5 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  20% Completed | 1/5 [00:03<00:15,  3.87s/it]
Loading safetensors checkpoint shards:  40% Completed | 2/5 [00:04<00:05,  1.99s/it]
Loading safetensors checkpoint shards:  60% Completed | 3/5 [00:09<00:06,  3.22s/it]
Loading safetensors checkpoint shards:  80% Completed | 4/5 [00:13<00:03,  3.74s/it]
Loading safetensors checkpoint shards: 100% Completed | 5/5 [00:18<00:00,  4.02s/it]
Loading safetensors checkpoint shards: 100% Completed | 5/5 [00:18<00:00,  3.66s/it]
(EngineCore_DP0 pid=41855) 


(EngineCore_DP0 pid=41855) INFO 01-25 17:00:17 [default_loader.py:291] Loading weights took 18.39 seconds
(EngineCore_DP0 pid=41855) INFO 01-25 17:00:19 [gpu_model_runner.py:3905] Model loading took 15.59 GiB memory and 30.040897 seconds
(EngineCore_DP0 pid=41855) WARNING 01-25 17:00:19 [fused_moe.py:1090] Using default MoE config. Performance might be sub-optimal! Config file not found at /data/ephemeral/home/8054_charmi/PTMT-GraphDB/.venv/lib/python3.10/site-packages/vllm/model_executor/layers/fused_moe/configs/E=128,N=768,device_name=Tesla_V100-SXM2-32GB,dtype=int4_w4a16.json
(EngineCore_DP0 pid=41855) INFO 01-25 17:00:37 [gpu_worker.py:358] Available KV cache memory: 11.53 GiB
(EngineCore_DP0 pid=41855) INFO 01-25 17:00:37 [kv_cache_utils.py:1305] GPU KV cache size: 125,888 tokens
(EngineCore_DP0 pid=41855) INFO 01-25 17:00:37 [kv_cache_utils.py:1310] Maximum concurrency for 2,048 tokens per request: 61.47x
(EngineCore_DP0 pid=41855) INFO 01-25 17:00:37 [core.py:273] init engine (p

In [19]:
ALIAS_PROMPT = """You are an expert entity normalizer.

Your task is to generate alias surface forms for a given canonical concept title.
These aliases are used to map different user-written strings to the SAME concept node
in a concept graph.

Return ONLY valid, raw JSON. No extra text. No markdown.

### Rules:
- "alias" MUST be a list of UNIQUE strings (no duplicates).
- Do NOT include the canonical title itself.
- Do NOT include trivial variants that are the same as the canonical title
  (case-only changes, extra spaces, or punctuation-only changes).
- If you cannot find at least 1 safe alias, output: {"alias": []}
- Provide 1 to 6 aliases max.

### What counts as an alias
Generate only realistic surface-form variants that people actually write:
- Case variants (upper/lower/mixed)
- Spacing variants (with or without spaces)
- Punctuation variants (hyphens, dots, slashes)
- Acronyms OR full names ONLY if they are unambiguous and commonly used
- Very common typos or misspellings (max 2–3)

Do NOT be creative. Be conservative.

### Examples (for pattern understanding only — do NOT copy)

Input: "machine learning"
Output: {"alias": ["ML", "MachineLearning", "machine-learning"]}

Input: "Wi-Fi"
Output: {"alias": ["wifi", "WiFi", "wi fi"]}

Input: "Seq2Seq"
Output: {"alias": ["seq2seq", "Seq2Seq", "sequence-to-sequence", "sequence to sequence"]}

### Task
Canonical Title: "{INPUT_TITLE}"
Output: """

def add_alias_column(node_df: pd.DataFrame) -> pd.DataFrame:
    df = node_df.copy()
    titles = df["name"].dropna().astype(str).unique().tolist()

    chat_inputs = []
    for title in tqdm(titles, desc="alias: build prompts"):
        system_msg = ALIAS_PROMPT.replace("{INPUT_TITLE}", title)
        user_msg = 'Return ONLY raw JSON in the form {"alias": [...]}'
        chat_inputs.append(build_chat_prompt(system_msg, user_msg))

    generations = llm.generate(chat_inputs, sampling)

    alias_map = {}
    raw_map = {}
    ok_map = {}

    for title, gen in tqdm(list(zip(titles, generations)), total=len(titles), desc="alias: parse outputs"):
        raw = gen.outputs[0].text.strip()
        raw_map[title] = raw

        parsed = extract_json(raw)
        ok = isinstance(parsed, dict) and isinstance(parsed.get("alias"), list)
        ok_map[title] = ok

        cleaned = []
        if ok:
            dedup = set()
            for item in parsed["alias"]:
                if not isinstance(item, str):
                    continue
                s = item.strip()
                if not s:
                    continue
                if s == title:   # canonical title 제외
                    continue
                if s not in dedup:
                    dedup.add(s)
                    cleaned.append(s)

        alias_map[title] = cleaned

    df["alias"] = df["name"].astype(str).map(alias_map)
    df["alias_raw"] = df["name"].astype(str).map(raw_map)
    df["alias_ok"] = df["name"].astype(str).map(ok_map)
    return df


In [20]:
preprocess_nodes = add_alias_column(nodes)

alias: parse outputs: 100%|██████████| 472/472 [00:00<00:00, 82104.82it/s]


In [21]:
preprocess_nodes

,name,link,alias,alias_raw,alias_ok
0,A priori probability,https://en.wikipedia.org/wiki/A_priori_probabi...,[a priori probability],"{""alias"": [""a priori probability"", ""a priori p...",True
1,AC-3 algorithm,https://en.wikipedia.org/wiki/AC-3_algorithm,"[ac-3, AC3, ac3, A C 3, ac 3]","{""alias"": [""ac-3"", ""AC3"", ""ac3"", ""A C 3"", ""ac ...",True
2,Abstract data type,https://en.wikipedia.org/wiki/Abstract_data_type,"[abstract data type, ADT, abstract datatype, a...","{""alias"": [""abstract data type"", ""ADT"", ""abstr...",True
3,Abstract type,https://en.wikipedia.org/wiki/Abstract_type,"[abstract type, AbstractType, abstract-type, a...","{""alias"": [""abstract type"", ""AbstractType"", ""a...",True
4,Adjusted mutual information,https://en.wikipedia.org/wiki/Adjusted_mutual_...,"[AMI, adjusted mutual information, ami, adjust...","{""alias"": [""AMI"", ""adjusted mutual information...",True
...,...,...,...,...,...
467,Well-quasi-ordering,https://en.wikipedia.org/wiki/Well-quasi-ordering,"[well quasi ordering, well-quasi ordering, wqo...","{""alias"": [""well quasi ordering"", ""well-quasi ...",True
468,Women in government,https://en.wikipedia.org/wiki/Women_in_government,"[women in gov, women in government, women in p...","{""alias"": [""women in gov"", ""women in governmen...",True
469,Y-fast trie,https://en.wikipedia.org/wiki/Y-fast_trie,"[y-fast trie, yfast trie, y-fast-trie, y fast ...","{""alias"": [""y-fast trie"", ""yfast trie"", ""y-fas...",True
470,Z-test,https://en.wikipedia.org/wiki/Z-test,"[z test, z-test, Z test, ztest, ZTest]","{""alias"": [""z test"", ""z-test"", ""Z test"", ""ztes...",True


In [22]:
preprocess_nodes.iloc[1]['alias']

['ac-3', 'AC3', 'ac3', 'A C 3', 'ac 3']

In [23]:
print(preprocess_nodes["alias_ok"].value_counts(dropna=False))
preprocess_nodes.loc[~preprocess_nodes["alias_ok"], ["name", "alias_raw"]].head(10)

alias_ok
True     471
False      1
Name: count, dtype: int64


,name,alias_raw
48,Christmas tree,"{""alias"": [""christmas tree"", ""ChristmasTree"", ..."


In [24]:
EDGE_PROMPT = """You are an expert curriculum designer and concept dependency evaluator.

Your task is to judge whether PREREQ is prerequisite knowledge for understanding TARGET.

Return ONLY valid JSON.
No explanations.
No markdown.
No extra text.

The output JSON must have exactly these fields:
{
  "strength": number (float between 0 and 1),
  "reason": string (1–2 short sentences, generic, no names or dates)
}

strength guidelines:
- 0.9–1.0: required to understand TARGET
- 0.6–0.8: very helpful but not required
- 0.3–0.5: weak background
- 0.0–0.2: not a prerequisite

If PREREQ is only loosely related, use strength <= 0.2.
If PREREQ is a subtype or example of TARGET, use strength <= 0.2.
If PREREQ is a foundational concept used to define or derive TARGET, use strength >= 0.6.

Example:
TARGET: Backpropagation
PREREQ: Chain Rule
Output:
{"strength":0.95,"reason":"Backpropagation relies on the chain rule to compute gradients through composed functions."}

Now evaluate:

TARGET: {TARGET}
PREREQ: {PREREQ}

Output:
"""

def add_edge_reason_strength(edge_df: pd.DataFrame) -> pd.DataFrame:
    df = edge_df.copy()
    key_df = df[["name", "prereq_name"]].dropna().drop_duplicates().astype(str)
    key_pairs = list(key_df.itertuples(index=False, name=None))

    chat_inputs = []
    for target, prereq in tqdm(key_pairs, desc="edge: build prompts"):
        system_msg = EDGE_PROMPT.replace("{TARGET}", target).replace("{PREREQ}", prereq)
        user_msg = "Return raw JSON only."
        chat_inputs.append(build_chat_prompt(system_msg, user_msg))

    generations = llm.generate(chat_inputs, sampling)

    # (target, prereq) -> (strength, reason, raw, ok)
    score_map: dict[tuple[str, str], tuple[object, object, str, bool]] = {}

    for (target, prereq), gen in tqdm(
        list(zip(key_pairs, generations)),
        total=len(key_pairs),
        desc="edge: parse outputs",
    ):
        raw = gen.outputs[0].text.strip()
        parsed = extract_json(raw)

        ok = isinstance(parsed, dict) and ("strength" in parsed or "reason" in parsed)

        strength = pd.NA
        reason = pd.NA

        if isinstance(parsed, dict):
            s = parsed.get("strength")
            r = parsed.get("reason")

            if isinstance(s, (int, float)):
                strength = float(max(0.0, min(1.0, s)))
            if isinstance(r, str) and r.strip():
                reason = r.strip()

        score_map[(target, prereq)] = (strength, reason, raw, ok)

    def _get(row, idx):
        return score_map.get((str(row["name"]), str(row["prereq_name"])),
                             (pd.NA, pd.NA, "", False))[idx]

    df["strength"] = df.apply(lambda row: _get(row, 0), axis=1)
    df["reason"]   = df.apply(lambda row: _get(row, 1), axis=1)
    df["edge_raw"] = df.apply(lambda row: _get(row, 2), axis=1)
    df["edge_ok"]  = df.apply(lambda row: _get(row, 3), axis=1)

    return df


In [25]:
preprocess_edges = add_edge_reason_strength(edges)

edge: parse outputs: 100%|██████████| 2848/2848 [00:00<00:00, 96925.41it/s]


In [26]:
preprocess_edges

,name,prereq_name,strength,reason,edge_raw,edge_ok
0,A priori probability,Erlang distribution,0.10,The Erlang distribution is unrelated to a prio...,"{""strength"":0.1,""reason"":""The Erlang distribut...",True
1,A priori probability,Gamma distribution,0.10,The gamma distribution is unrelated to the con...,"{""strength"":0.1,""reason"":""The gamma distributi...",True
2,A priori probability,Inverse-gamma distribution,0.10,The inverse-gamma distribution is unrelated to...,"{""strength"":0.1,""reason"":""The inverse-gamma di...",True
3,A priori probability,Normal-gamma distribution,0.10,The normal-gamma distribution is a specific pr...,"{""strength"":0.1,""reason"":""The normal-gamma dis...",True
4,AC-3 algorithm,Asymptotic computational complexity,0.40,Understanding asymptotic computational complex...,"{""strength"":0.4,""reason"":""Understanding asympt...",True
...,...,...,...,...,...,...
2843,Z-test,Uniform distribution (continuous),0.10,The Z-test does not rely on the uniform distri...,"{""strength"":0.1,""reason"":""The Z-test does not ...",True
2844,Z-test,Variance,0.85,Variance is essential for calculating the stan...,"{""strength"":0.85,""reason"":""Variance is essenti...",True
2845,Zero of a function,Complex analysis,0.20,Complex analysis involves zeros of functions b...,"{""strength"":0.2,""reason"":""Complex analysis inv...",True
2846,Zero of a function,Complex number,0.20,Complex numbers are not required to understand...,"{""strength"":0.2,""reason"":""Complex numbers are ...",True


In [27]:
preprocess_edges.iloc[1]['reason']

'The gamma distribution is unrelated to the concept of a priori probability, which refers to initial beliefs before evidence is considered.'

In [28]:
print(preprocess_edges["edge_ok"].value_counts(dropna=False))
preprocess_edges.loc[~preprocess_edges["edge_ok"], ["name", "edge_raw"]].head(10)

edge_ok
True    2848
Name: count, dtype: int64


,name,edge_raw


In [29]:
preprocess_edges['strength'].describe()

count    2848.000000
mean        0.547647
std         0.290382
min         0.000000
25%         0.300000
50%         0.700000
75%         0.850000
max         0.950000
Name: strength, dtype: float64

### Node 용 데이터셋 저장

In [30]:
## name, link, alias

preprocess_nodes[['name', 'link', 'alias']].to_csv('../../data/kc_dataset/metacademy/metacademy_node.csv', index=False)

### Edge 용 데이터셋 저장

In [31]:
## name, prereq_name, strength, reason

preprocess_edges[['name', 'prereq_name', 'strength', 'reason']].to_csv('../../data/kc_dataset/metacademy/metacademy_edge.csv', index=False)